In [4]:
import pandas as pd
dataset=pd.read_json('/home/hammadali08/Personal/FYP Datasets/News_Category_Dataset_v3.json',lines=True)
dataset=dataset.head(1000)

In [5]:
dataset.shape

(1000, 6)

In [6]:
from nltk.stem import WordNetLemmatizer
import nltk
import re
import numpy as np
import gensim
from nltk.corpus import stopwords
lemmatizer = WordNetLemmatizer()

In [7]:
corpus = []
for i in range(0, len(dataset)):
    review = re.sub('[^a-zA-Z0-9]', ' ', dataset['headline'][i])
    review = review.lower()
    review = review.split()

    review = [lemmatizer.lemmatize(word) for word in review if word not in set(stopwords.words('english'))]
    review = ' '.join(review)
    corpus.append(review)

In [34]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess
words=[]
for sent in corpus:
    sent_token=sent_tokenize(sent)
    for sent in sent_token:
        words.append(simple_preprocess(sent))

In [37]:
## Lets train Word2vec from scratch
model=gensim.models.Word2Vec(words,window=5,min_count=2)

In [38]:
def avg_word2vec(doc):
    # remove out-of-vocabulary words
    #sent = [word for word in doc if word in model.wv.index_to_key]
    #print(sent)

    return np.mean([model.wv[word] for word in doc if word in model.wv.index_to_key], axis=0)
    #or [np.zeros(len(model.wv.index_to_key))], axis=0)


from tqdm import tqdm
#apply for the entire sentences
import numpy as np

X = []
for i in tqdm(range(len(words))):
    X.append(avg_word2vec(words[i]))
X

  0%|          | 0/1000 [00:00<?, ?it/s]/home/hammadali08/.local/lib/python3.12/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/hammadali08/.local/lib/python3.12/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
100%|██████████| 1000/1000 [00:00<00:00, 7916.86it/s]


[array([-1.6921196e-03,  2.0784950e-03,  3.8814349e-03, -2.7629535e-03,
         9.3014132e-05,  9.6167298e-04, -2.2096403e-03,  2.9958428e-03,
         4.6977033e-03, -2.8722312e-03, -2.1449395e-03, -4.2890650e-03,
        -1.6617781e-03,  1.0750800e-03, -1.2179512e-03, -1.9372813e-03,
        -3.2389434e-03, -1.2493186e-03,  1.2771926e-03,  1.2484988e-03,
         2.6109123e-03,  1.5733488e-03,  4.4209821e-04,  2.2072280e-03,
         2.1083672e-03, -1.4151200e-03,  4.9952767e-04,  5.8912984e-03,
         1.4522159e-03, -2.6346141e-04,  1.1347936e-03,  1.4072629e-03,
        -1.8243060e-03,  5.9361430e-04,  1.5076337e-03, -3.7120797e-03,
         2.6054625e-04, -8.5458643e-04,  6.5410143e-04, -1.4623461e-03,
        -2.4656106e-03, -7.0436019e-04, -3.1714675e-03,  2.2772143e-03,
        -3.7160746e-04, -1.7235487e-03, -9.3436847e-04, -1.3909076e-03,
         5.6403875e-03,  2.0921221e-03,  2.8710980e-03, -1.8718028e-03,
         7.7804155e-04, -2.4812913e-03,  4.0624980e-03,  2.53843

In [60]:
df_list = []
for i in range(len(X)):
    df_list.append(pd.DataFrame(X[i].reshape(1, -1)))

df = pd.concat(df_list, ignore_index=True)
df

/tmp/ipykernel_40536/578229423.py:5: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(df_list, ignore_index=True)


,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,-0.001692,0.002078,0.003881,-0.002763,0.000093,0.000962,-0.002210,0.002996,0.004698,-0.002872,...,-0.001016,-0.002916,0.000420,-0.001395,0.000319,0.000793,0.002543,0.001569,-0.000474,-0.000783
1,0.001984,-0.003748,0.002595,-0.001830,0.001207,-0.004720,-0.000447,-0.000549,-0.002535,-0.001958,...,0.001233,-0.001813,0.001182,0.000521,-0.002001,-0.000541,0.003293,0.000268,-0.000971,0.000819
2,0.006186,0.005968,0.001846,-0.002043,0.002111,-0.003224,0.000028,0.005039,-0.001531,0.001657,...,0.001648,0.001808,0.002272,0.000299,0.003013,-0.001435,-0.004028,-0.001537,0.000021,-0.001059
3,0.005072,0.005662,-0.000631,-0.004102,-0.000116,-0.001234,0.000396,0.003832,0.000597,0.001762,...,-0.000063,0.001985,0.007135,-0.000645,0.003497,0.001635,-0.002556,0.001136,0.000777,-0.000708
4,0.000549,-0.000466,-0.002418,0.000432,-0.000767,0.000330,0.002432,0.001347,-0.000619,-0.002574,...,0.002055,0.001516,-0.002083,0.004471,0.000892,-0.000859,0.001000,-0.003371,0.001252,0.003628
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,-0.000224,0.005231,0.001365,-0.001957,0.000204,0.002336,-0.001827,0.005380,0.002253,-0.000982,...,0.001659,-0.000983,0.005122,-0.001444,0.000293,0.002726,0.000942,0.000421,-0.000267,-0.002971
996,-0.003956,0.002360,0.000243,-0.002932,-0.001079,-0.004802,-0.000447,0.001851,-0.002673,-0.000905,...,-0.000659,0.001369,-0.002304,0.002567,0.003615,-0.001689,0.004574,0.000007,-0.001543,-0.003086
997,0.000249,0.001959,0.004581,-0.001466,0.001937,-0.002239,0.000344,-0.001580,-0.000349,-0.004969,...,-0.000142,0.001420,-0.002876,-0.000320,0.001499,0.000786,-0.001490,0.000840,0.000955,-0.005181
998,0.004997,-0.002942,0.003928,0.001445,-0.001802,-0.004976,-0.001886,0.001343,-0.002747,-0.001113,...,0.002659,0.004035,0.001571,-0.003239,-0.000814,0.004924,0.001295,-0.000343,0.001765,-0.000232


# Output Preprocessing

In [35]:
corpus1 = []
for i in range(0, len(dataset)):
    review1 = re.sub('[^a-zA-Z0-9]', ' ', dataset['short_description'][i])
    review1 = review1.lower()
    review1 = review1.split()

    review1 = [lemmatizer.lemmatize(word) for word in review1 if word not in set(stopwords.words('english'))]
    review1 = ' '.join(review1)
    corpus1.append(review1)

In [36]:
decoder_input_data = ["<start> " + txt + "" for txt in corpus1]
decoder_target_data= ["" + txt + " <end>" for txt in corpus1]
decoder_input_data

['<start> health expert said early predict whether demand would match 171 million dos new booster u ordered fall',
 '<start> subdued passenger crew fled back aircraft confrontation according u attorney office los angeles',
 '<start> dog understand could eaten',
 '<start> accidentally put grown toothpaste toddler toothbrush screamed like cleaning teeth carolina reaper dipped tabasco sauce',
 '<start> amy cooper accused investment firm franklin templeton unfairly firing branding racist video central park encounter went viral',
 '<start> 63 year old woman seen working south carolina store thursday found dead monday family reported missing authority said',
 '<start> behind anchor new york pix11 asked journalist michelle ross finished interview',
 '<start> half million people remained without water service three day storm lashed u territory',
 '<start> mija director isabel castro combined music documentary style euphoria clueless tell nuanced immigration story',
 '<start> white house offici

In [45]:
words_output_input=[]
for sent in decoder_input_data:
    sent_token=sent_tokenize(sent)
    for sent in sent_token:
        words_output_input.append(simple_preprocess(sent))

In [46]:
words_output_target=[]
for sent in decoder_target_data:
    sent_token=sent_tokenize(sent)
    for sent in sent_token:
        words_output_target.append(simple_preprocess(sent))

In [47]:
model_output=gensim.models.Word2Vec(words_output_input,window=5,min_count=2)

In [48]:
def avg_word2vec_output(doc_output):
    # remove out-of-vocabulary words
    #sent = [word for word in doc if word in model.wv.index_to_key]
    #print(sent)

    return np.mean([model_output.wv[word]for word in doc_output if word in model_output.wv.index_to_key],axis=0)
                           #or [np.zeros(len(model.wv.index_to_key))], axis=0)

In [50]:
Y_decoder_input_data = []
for i in tqdm(range(len(words_output_input))):
    Y_decoder_input_data.append(avg_word2vec_output(words_output_input[i]))
Y_decoder_input_data

100%|██████████| 1000/1000 [00:00<00:00, 4721.33it/s]


[array([-0.00995481,  0.01255069,  0.00248424,  0.00274645, -0.00113236,
        -0.02029816,  0.00578677,  0.02837395, -0.01379669, -0.010314  ,
        -0.00523877, -0.01965367,  0.00021725,  0.00293196,  0.0085965 ,
        -0.01065332,  0.00719154, -0.0132163 ,  0.00054096, -0.02466753,
         0.00170329,  0.00599145,  0.01076173, -0.00994472, -0.00306757,
        -0.0088687 , -0.01779817, -0.00885704, -0.01168091,  0.00640739,
         0.0080679 ,  0.0018776 ,  0.00486152, -0.01260315, -0.00843052,
         0.01799867,  0.00155224, -0.01707185, -0.00361557, -0.02357236,
        -0.00441941, -0.00929202, -0.00460561,  0.00399182,  0.009019  ,
        -0.00628071, -0.0147532 , -0.0043939 ,  0.00432628,  0.00953387,
         0.00455845, -0.00586212, -0.00689808,  0.00257978, -0.00991385,
         0.00833921,  0.00818354, -0.0064026 , -0.0169968 ,  0.00773275,
         0.00256443,  0.00278742, -0.00327614, -0.00128959, -0.0190476 ,
         0.0112001 , -0.00182214,  0.00719454, -0.0

In [51]:
Y_decoder_target_data = []
for i in tqdm(range(len(words_output_target))):
    Y_decoder_target_data.append(avg_word2vec_output(words_output_target[i]))
Y_decoder_target_data

100%|██████████| 1000/1000 [00:00<00:00, 4738.86it/s]


[array([-7.38353841e-03,  9.78512689e-03,  2.03370326e-03,  2.25377199e-03,
        -6.16411737e-04, -1.53670413e-02,  4.48533148e-03,  2.08123401e-02,
        -1.04866019e-02, -8.75060167e-03, -5.09402761e-03, -1.55330729e-02,
         6.68313369e-05,  2.59572524e-03,  6.99929893e-03, -8.25479720e-03,
         5.37188118e-03, -9.86631680e-03, -1.75673762e-04, -1.75207816e-02,
         3.89258028e-04,  3.91151989e-03,  7.25405384e-03, -8.71638861e-03,
        -2.67217797e-03, -7.30671734e-03, -1.37819434e-02, -6.74920017e-03,
        -8.33642855e-03,  5.34839788e-03,  6.90716552e-03,  1.38578692e-03,
         3.31836985e-03, -9.38469451e-03, -6.71511702e-03,  1.43838078e-02,
         1.93268096e-03, -1.34091955e-02, -2.45600333e-03, -1.75047759e-02,
        -2.43999693e-03, -7.55551225e-03, -2.54391856e-03,  3.25594074e-03,
         8.11898522e-03, -4.37058415e-03, -1.12003377e-02, -3.98336444e-03,
         3.46148154e-03,  6.13835920e-03,  3.96363111e-03, -4.51035658e-03,
        -5.1

In [62]:
df_list_ = []
for i in range(len(Y_decoder_input_data)):
    df_list_.append(pd.DataFrame(Y_decoder_input_data[i].reshape(1, -1)))

df_ = pd.concat(df_list_, ignore_index=True)
df_

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,-0.009955,0.012551,0.002484,0.002746,-0.001132,-0.020298,0.005787,0.028374,-0.013797,-0.010314,...,0.019160,0.010345,0.008425,0.000238,0.022699,0.012283,0.002200,-0.019728,-0.003216,0.000955
1,-0.007188,0.009979,0.005460,0.005137,-0.005479,-0.018264,0.005167,0.023589,-0.010810,-0.003347,...,0.019381,0.008575,0.006574,-0.001775,0.020104,0.006338,-0.002134,-0.020095,-0.000494,0.001148
2,-0.018539,0.017843,0.006643,0.001408,-0.001963,-0.037031,0.010898,0.042010,-0.024699,-0.013543,...,0.033260,0.012286,0.014459,-0.000773,0.039867,0.015286,0.001927,-0.029340,-0.006023,-0.004346
3,-0.012680,0.015274,0.004179,0.005995,0.000391,-0.023540,0.003925,0.034456,-0.019633,-0.007359,...,0.021613,0.012746,0.007558,0.001873,0.028990,0.017280,0.004331,-0.024007,-0.002394,0.000957
4,-0.007546,0.006125,0.002688,0.005567,-0.002799,-0.018624,0.003058,0.019020,-0.011971,-0.005116,...,0.014329,0.005065,0.006072,0.000188,0.015320,0.008009,0.003343,-0.015646,-0.002086,0.004094
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,-0.006824,0.009871,0.006499,0.001674,-0.002281,-0.016478,0.007642,0.018748,-0.010715,-0.004472,...,0.018645,0.006083,0.007192,0.000145,0.018403,0.009782,0.004868,-0.017317,0.000694,-0.000169
996,-0.012747,0.012922,0.002222,0.002085,0.000298,-0.025251,0.007296,0.030650,-0.013960,-0.009967,...,0.022810,0.007603,0.007809,0.002342,0.025725,0.012830,0.003111,-0.023820,0.000172,0.000663
997,-0.011331,0.013144,0.003116,0.005349,-0.002119,-0.022164,0.004783,0.029302,-0.016676,-0.003722,...,0.020390,0.006513,0.007725,-0.002829,0.022800,0.011216,0.003407,-0.018849,-0.003032,-0.002001
998,-0.006830,0.008778,0.004206,0.005936,0.002437,-0.016957,0.003999,0.021440,-0.014588,-0.002215,...,0.016114,0.008240,0.010023,0.003568,0.019607,0.011270,0.005063,-0.020272,-0.001103,-0.002642


In [64]:
df_list_ = []
for i in range(len(Y_decoder_input_data)):
    df_list_.append(pd.DataFrame(Y_decoder_input_data[i].reshape(1, -1)))

df_target = pd.concat(df_list_, ignore_index=True)
df_target

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,-0.009955,0.012551,0.002484,0.002746,-0.001132,-0.020298,0.005787,0.028374,-0.013797,-0.010314,...,0.019160,0.010345,0.008425,0.000238,0.022699,0.012283,0.002200,-0.019728,-0.003216,0.000955
1,-0.007188,0.009979,0.005460,0.005137,-0.005479,-0.018264,0.005167,0.023589,-0.010810,-0.003347,...,0.019381,0.008575,0.006574,-0.001775,0.020104,0.006338,-0.002134,-0.020095,-0.000494,0.001148
2,-0.018539,0.017843,0.006643,0.001408,-0.001963,-0.037031,0.010898,0.042010,-0.024699,-0.013543,...,0.033260,0.012286,0.014459,-0.000773,0.039867,0.015286,0.001927,-0.029340,-0.006023,-0.004346
3,-0.012680,0.015274,0.004179,0.005995,0.000391,-0.023540,0.003925,0.034456,-0.019633,-0.007359,...,0.021613,0.012746,0.007558,0.001873,0.028990,0.017280,0.004331,-0.024007,-0.002394,0.000957
4,-0.007546,0.006125,0.002688,0.005567,-0.002799,-0.018624,0.003058,0.019020,-0.011971,-0.005116,...,0.014329,0.005065,0.006072,0.000188,0.015320,0.008009,0.003343,-0.015646,-0.002086,0.004094
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,-0.006824,0.009871,0.006499,0.001674,-0.002281,-0.016478,0.007642,0.018748,-0.010715,-0.004472,...,0.018645,0.006083,0.007192,0.000145,0.018403,0.009782,0.004868,-0.017317,0.000694,-0.000169
996,-0.012747,0.012922,0.002222,0.002085,0.000298,-0.025251,0.007296,0.030650,-0.013960,-0.009967,...,0.022810,0.007603,0.007809,0.002342,0.025725,0.012830,0.003111,-0.023820,0.000172,0.000663
997,-0.011331,0.013144,0.003116,0.005349,-0.002119,-0.022164,0.004783,0.029302,-0.016676,-0.003722,...,0.020390,0.006513,0.007725,-0.002829,0.022800,0.011216,0.003407,-0.018849,-0.003032,-0.002001
998,-0.006830,0.008778,0.004206,0.005936,0.002437,-0.016957,0.003999,0.021440,-0.014588,-0.002215,...,0.016114,0.008240,0.010023,0.003568,0.019607,0.011270,0.005063,-0.020272,-0.001103,-0.002642


In [21]:
df_.dropna(inplace=True)
df_.shape

(998, 100)